In [0]:
# ============================================================
# 04_ML_TRAINING
# Gold → Feature prep → Train model → Log with MLflow
# ============================================================

GOLD_TABLE = "workspace.default.gold_loan_features"

df = spark.table(GOLD_TABLE)
print("✅ Rows from Gold:", df.count())
print("✅ Columns:", len(df.columns))

✅ Rows from Gold: 307511
✅ Columns: 89


In [0]:
# Select the best features for training
feature_cols = [
    "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE",
    "CODE_GENDER", "FLAG_OWN_CAR", "FLAG_OWN_REALTY",
    "CNT_CHILDREN", "CNT_FAM_MEMBERS",
    "DAYS_BIRTH", "DAYS_EMPLOYED",
    "AGE_YEARS", "EMPLOYMENT_YEARS", "IS_UNEMPLOYED",
    "CREDIT_INCOME_RATIO", "ANNUITY_INCOME_RATIO",
    "CREDIT_GOODS_RATIO", "INCOME_PER_PERSON",
    "EXT_SOURCE_2", "EXT_SOURCE_3", "EXT_SOURCE_MEAN", "EXT_SOURCE_MIN",
    "EMPLOYMENT_TO_AGE_RATIO", "TOTAL_DOCS_SUBMITTED",
    "SOCIAL_CIRCLE_DEFAULT_RATE",
    "REGION_RATING_CLIENT", "REGION_RATING_CLIENT_W_CITY",
    "FLAG_WORK_PHONE", "FLAG_PHONE", "FLAG_EMAIL",
    "AMT_REQ_CREDIT_BUREAU_YEAR", "AMT_REQ_CREDIT_BUREAU_MON"
]

TARGET_COL = "TARGET"

# Convert to Pandas for Scikit-learn
pdf = df.select(feature_cols + [TARGET_COL]).toPandas()
print("✅ Shape:", pdf.shape)
print("✅ Target distribution:\n", pdf[TARGET_COL].value_counts())

✅ Shape: (307511, 33)
✅ Target distribution:
 TARGET
0    282686
1     24825
Name: count, dtype: int64


In [0]:
from sklearn.model_selection import train_test_split

X = pdf[feature_cols]
y = pdf[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("✅ Train size:", X_train.shape)
print("✅ Test size:", X_test.shape)
print("✅ Train target distribution:\n", y_train.value_counts())

✅ Train size: (246008, 32)
✅ Test size: (61503, 32)
✅ Train target distribution:
 TARGET
0    226148
1     19860
Name: count, dtype: int64


In [0]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, roc_auc_score,
                              f1_score, precision_score, recall_score,
                              classification_report)
from mlflow.models.signature import infer_signature

class_weights = {0: 1, 1: 10}

mlflow.set_experiment("/loan_default_prediction")

with mlflow.start_run(run_name="RandomForest_v1"):

    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=8,
        class_weight=class_weights,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    # Infer signature — required for Unity Catalog
    signature = infer_signature(X_train, model.predict(X_train))

    acc  = accuracy_score(y_test, y_pred)
    auc  = roc_auc_score(y_test, y_prob)
    f1   = f1_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)

    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 8)
    mlflow.log_param("class_weight", str(class_weights))
    mlflow.log_param("train_size", X_train.shape[0])

    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("roc_auc", auc)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("precision", prec)
    mlflow.log_metric("recall", rec)

    # Log model WITH signature
    mlflow.sklearn.log_model(
        model,
        "random_forest_model",
        signature=signature
    )

    print("=" * 50)
    print("✅ MODEL TRAINED & LOGGED TO MLFLOW")
    print("=" * 50)
    print(f"Accuracy  : {acc:.4f}")
    print(f"ROC-AUC   : {auc:.4f}")
    print(f"F1 Score  : {f1:.4f}")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/06/15 18:07:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-f4a3552c-72d6.cloud.databricks.com/ml/experiments/2234382880014879/models/m-33882e00cf1144158e55a

✅ MODEL TRAINED & LOGGED TO MLFLOW
Accuracy  : 0.7199
ROC-AUC   : 0.7345
F1 Score  : 0.2611
Precision : 0.1659
Recall    : 0.6129

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.73      0.83     56538
           1       0.17      0.61      0.26      4965

    accuracy                           0.72     61503
   macro avg       0.56      0.67      0.54     61503
weighted avg       0.89      0.72      0.78     61503



In [0]:
import pandas as pd

# Top 15 most important features
feat_imp = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False).head(15)

print("📊 Top 15 Feature Importances:")
print(feat_imp.to_string(index=False))

📊 Top 15 Feature Importances:
                feature  importance
        EXT_SOURCE_MEAN    0.213324
         EXT_SOURCE_MIN    0.186851
           EXT_SOURCE_2    0.142767
           EXT_SOURCE_3    0.135712
     CREDIT_GOODS_RATIO    0.042422
             DAYS_BIRTH    0.036131
              AGE_YEARS    0.031644
          DAYS_EMPLOYED    0.027016
       EMPLOYMENT_YEARS    0.021965
        AMT_GOODS_PRICE    0.020493
EMPLOYMENT_TO_AGE_RATIO    0.017623
            CODE_GENDER    0.016794
             AMT_CREDIT    0.014635
            AMT_ANNUITY    0.014314
   ANNUITY_INCOME_RATIO    0.011846


In [0]:
# Register the model in MLflow Model Registry
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Get latest run id
experiment = client.get_experiment_by_name("/loan_default_prediction")
runs = client.search_runs(experiment.experiment_id, order_by=["metrics.roc_auc DESC"])
best_run_id = runs[0].info.run_id

print(f"✅ Best Run ID: {best_run_id}")
print(f"✅ Best ROC-AUC: {runs[0].data.metrics['roc_auc']:.4f}")

# Register model
model_uri = f"runs:/{best_run_id}/random_forest_model"
mv = mlflow.register_model(model_uri, "LoanDefaultModel")
print(f"✅ Model registered: LoanDefaultModel v{mv.version}")

✅ Best Run ID: 3114dc8d8f4c4988b2b014b51f23ef98
✅ Best ROC-AUC: 0.7345


Registered model 'LoanDefaultModel' already exists. Creating a new version of this model...
2026/06/15 18:08:49 WARNING mlflow.tracking._model_registry.fluent: Run with id 3114dc8d8f4c4988b2b014b51f23ef98 has no artifacts at artifact path 'random_forest_model', registering model based on models:/m-33882e00cf1144158e55a1a957e051c0 instead


Uploading artifacts:   0%|          | 0/10 [00:00<?, ?it/s]

🔗 Created version '1' of model 'workspace.default.loandefaultmodel': https://dbc-f4a3552c-72d6.cloud.databricks.com/explore/data/models/workspace/default/loandefaultmodel/version/1?o=7474644166959834


✅ Model registered: LoanDefaultModel v1


In [0]:
from pyspark.sql.functions import current_timestamp, lit
import pandas as pd

# Add predictions back
X_test_copy = X_test.copy()
X_test_copy["TARGET"] = y_test.values
X_test_copy["PREDICTED_DEFAULT"] = y_pred
X_test_copy["DEFAULT_PROBABILITY"] = y_prob.round(4)

# Convert to Spark
df_predictions = spark.createDataFrame(X_test_copy)
df_predictions = df_predictions \
    .withColumn("_predicted_at", current_timestamp()) \
    .withColumn("_model_version", lit("RandomForest_v1"))

# Save predictions table
PRED_TABLE = "workspace.default.gold_loan_predictions"
df_predictions.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(PRED_TABLE)

print(f"✅ Predictions saved to: {PRED_TABLE}")
print(f"✅ Total predictions: {df_predictions.count()}")

✅ Predictions saved to: workspace.default.gold_loan_predictions
✅ Total predictions: 61503
